# P1 — PREDICT

## Goal
Run the same sealed CLI as the package README, with a fresh Python process per stage. This notebook is a launcher, not an alternative implementation.

## Setup
Use the package root as the notebook working directory. Select its documented numerical Python environment, and set `P1_DATA_DIR` to the organizer dataset. Do not put raw data into upload ZIPs.

## Key assumptions
Training must start in a new empty output path. Do not erase models, logs or consumed locks to rerun. Inference requires the package's successful internal QA. No upload or final model lock occurs here. Execution logs stay local.

Validation status: generated launcher; structural/synthetic checks are separate from actual whole-cold numerical execution. Consult the package receipt for the latter.

In [ ]:
import os
import subprocess
import sys
from datetime import UTC, datetime
from pathlib import Path

package_root = Path.cwd().resolve()
assert (package_root / 'README.md').is_file(), 'Open from the extracted package root'
assert os.environ.get('P1_DATA_DIR'), 'Set P1_DATA_DIR first'
assert Path(os.environ['P1_DATA_DIR']).is_dir(), 'Dataset directory missing'
print({'python': sys.version.split()[0], 'role': 'PREDICT'})


## Steps
The next cell executes real work. Do not rerun a consumed training attempt.

In [ ]:
commands = [['02_code/run.py', 'infer'], ['02_code/run.py', 'verify']]
log_dir = package_root / '06_docs' / 'notebook_launches'
log_dir.mkdir(parents=True, exist_ok=True)
for index, args in enumerate(commands):
    entry = (package_root / args[0]).resolve()
    assert entry.is_relative_to(package_root) and entry.is_file(), 'Missing local entrypoint'
    stamp = datetime.now(UTC).strftime('%Y%m%dT%H%M%S%fZ')
    log = log_dir / ('predict_' + stamp + '_' + str(index) + '.log')
    with log.open('x', encoding='utf-8') as output:
        result = subprocess.run([sys.executable, '-I', '-B', *args], cwd=package_root, stdout=output, stderr=subprocess.STDOUT, check=False)
    print({'stage': index + 1, 'returncode': result.returncode, 'log': log.name})
    if result.returncode != 0:
        raise RuntimeError('Stage failed; preserve its outputs and inspect its log. Do not restart automatically.')


## Checks and next steps
A zero exit code is not a leaderboard score. Check the package's training/QA/answer receipt, row count, SHA and runtime. Preserve fallback files. Use only the exact validated CSV on its matching problem page; final model designation is separate.